# 🏦 US Tech Financial Intelligence Platform
**Accounting Data Analytics with Python @ UIUC Coursera**

Author: Wayne | Tools: Python · Pandas · SQLite · Plotly · Matplotlib

---
## 📦 SECTION 1 — Setup

In [ ]:
!pip install yfinance pandas numpy sqlalchemy plotly matplotlib -q

In [ ]:
import yfinance as yf
import pandas as pd, numpy as np, sqlite3
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'axes.facecolor':'#0E1117','figure.facecolor':'#0A0C10',
    'text.color':'#E8EAF0','axes.labelcolor':'#9AA0B0','xtick.color':'#9AA0B0',
    'ytick.color':'#9AA0B0','axes.edgecolor':'#1E2330','grid.color':'#1E2330'})

GOLD,BLUE,GREEN,RED,PURPLE = '#C9A84C','#4A9EFF','#2ECC71','#E05252','#A78BFA'
BG,SURFACE = '#0A0C10','#111318'
print('✅ Ready:', datetime.now().strftime('%Y-%m-%d %H:%M'))

---
## 🗂️ SECTION 2 — Data Pipeline

In [ ]:
STOCKS = {
    'Apple':'AAPL','Microsoft':'MSFT','Google':'GOOGL',
    'Meta':'META','Amazon':'AMZN','NVIDIA':'NVDA','TSMC':'TSM'
}
for k,v in STOCKS.items(): print(f'  {v:8s} -> {k}')

In [ ]:
price_data = {}
for name, ticker in STOCKS.items():
    try:
        h = yf.Ticker(ticker).history(period='3y')
        h['Ticker']=ticker; h['Company']=name
        price_data[ticker] = h
        print(f'✅ {name} ({ticker}) — {len(h)} rows')
    except Exception as e:
        print(f'❌ {name}: {e}')
df_prices = pd.concat(price_data.values()).reset_index()
df_prices.rename(columns={'index':'Date'}, inplace=True)
print(f'\nTotal: {len(df_prices)} rows')

In [ ]:
def fetch(ticker, name):
    s = yf.Ticker(ticker); r = {}
    for key, attr in [('income','quarterly_financials'),
                       ('balance','quarterly_balance_sheet'),
                       ('cashflow','quarterly_cashflow')]:
        try:
            df = getattr(s, attr).T
            df['Ticker']=ticker; df['Company']=name
            r[key]=df; print(f'  ✅ {name} {key}')
        except Exception as e:
            print(f'  ❌ {name} {key}: {e}')
    return r

all_fin = {}
for name, ticker in STOCKS.items():
    print(f'\n📥 {name}...')
    all_fin[ticker] = fetch(ticker, name)
print('\n✅ Done')

In [ ]:
rows = []
for ticker, data in all_fin.items():
    if 'income' not in data: continue
    df = data['income'].copy()
    df.index = pd.to_datetime(df.index); df = df.sort_index()
    want = ['Total Revenue','Gross Profit','Operating Income','Net Income','EBITDA','Ticker','Company']
    df = df[[c for c in want if c in df.columns]]
    for col in ['Total Revenue','Gross Profit','Operating Income','Net Income','EBITDA']:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce') / 1e9
    df['Quarter'] = df.index.to_period('Q').astype(str)
    rows.append(df)
df_income = pd.concat(rows).reset_index().rename(columns={'index':'Date'})
df_income['Gross_Margin'] = (df_income['Gross Profit'] / df_income['Total Revenue'] * 100).round(2)
df_income['Net_Margin']   = (df_income['Net Income']   / df_income['Total Revenue'] * 100).round(2)
df_income['Op_Margin']    = (df_income['Operating Income'] / df_income['Total Revenue'] * 100).round(2)
df_income = df_income.sort_values(['Ticker','Date'])
df_income['Revenue_YoY'] = df_income.groupby('Ticker')['Total Revenue'].pct_change(4).mul(100).round(2)
print('✅ Income statement ready')

In [ ]:
rows = []
for ticker, data in all_fin.items():
    if 'balance' not in data: continue
    df = data['balance'].copy()
    df.index = pd.to_datetime(df.index); df = df.sort_index()
    want = ['Total Assets','Total Liabilities Net Minority Interest',
            'Stockholders Equity','Total Debt','Cash And Cash Equivalents','Ticker','Company']
    df = df[[c for c in want if c in df.columns]]
    for col in df.columns:
        if col not in ['Ticker','Company']:
            df[col] = pd.to_numeric(df[col], errors='coerce') / 1e9
    df['Quarter'] = df.index.to_period('Q').astype(str)
    rows.append(df)
df_balance = pd.concat(rows).reset_index().rename(columns={'index':'Date'})
if 'Total Liabilities Net Minority Interest' in df_balance.columns:
    df_balance['Debt_Ratio'] = (df_balance['Total Liabilities Net Minority Interest'] / df_balance['Total Assets'] * 100).round(2)
print('✅ Balance sheet ready')

In [ ]:
rows = []
for ticker, data in all_fin.items():
    if 'cashflow' not in data: continue
    df = data['cashflow'].copy()
    df.index = pd.to_datetime(df.index); df = df.sort_index()
    want = ['Operating Cash Flow','Capital Expenditure','Free Cash Flow','Ticker','Company']
    df = df[[c for c in want if c in df.columns]]
    for col in df.columns:
        if col not in ['Ticker','Company']:
            df[col] = pd.to_numeric(df[col], errors='coerce') / 1e9
    if 'Free Cash Flow' not in df.columns and 'Operating Cash Flow' in df.columns and 'Capital Expenditure' in df.columns:
        df['Free Cash Flow'] = df['Operating Cash Flow'] + df['Capital Expenditure']
    df['Quarter'] = df.index.to_period('Q').astype(str)
    rows.append(df)
df_cashflow = pd.concat(rows).reset_index().rename(columns={'index':'Date'})
print('✅ Cash flow ready')

In [ ]:
DB_PATH = 'financial_intelligence.db'
conn = sqlite3.connect(DB_PATH)
df_prices.to_sql('stock_prices',   conn, if_exists='replace', index=False)
df_income.to_sql('income_stmt',    conn, if_exists='replace', index=False)
df_balance.to_sql('balance_sheet', conn, if_exists='replace', index=False)
df_cashflow.to_sql('cash_flow',    conn, if_exists='replace', index=False)
for t in ['stock_prices','income_stmt','balance_sheet','cash_flow']:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0,0]
    print(f'  {t:20s}: {n} rows')
conn.close()
print('✅ SQLite saved')

---
## 📊 SECTION 3 — Financial Analysis

In [ ]:
ai = df_income[df_income['Ticker']=='AAPL'].sort_values('Date')
ab = df_balance[df_balance['Ticker']=='AAPL'].sort_values('Date')
ac = df_cashflow[df_cashflow['Ticker']=='AAPL'].sort_values('Date')
print('='*65); print('APPLE — Income Statement (USD Bn)'); print('='*65)
c1=[c for c in ['Quarter','Total Revenue','Gross Profit','Net Income','Gross_Margin','Net_Margin','Revenue_YoY'] if c in ai.columns]
print(ai[c1].tail(8).to_string(index=False))
print('\n'+'='*65); print('APPLE — Balance Sheet (USD Bn)'); print('='*65)
c2=[c for c in ['Quarter','Total Assets','Total Debt','Stockholders Equity','Debt_Ratio'] if c in ab.columns]
print(ab[c2].tail(5).to_string(index=False))
print('\n'+'='*65); print('APPLE — Cash Flow (USD Bn)'); print('='*65)
c3=[c for c in ['Quarter','Operating Cash Flow','Capital Expenditure','Free Cash Flow'] if c in ac.columns]
print(ac[c3].tail(8).to_string(index=False))

In [ ]:
dfm = pd.merge(
    df_income[['Date','Ticker','Company','Quarter','Net Income','Total Revenue','Gross_Margin','Net_Margin','Op_Margin']],
    df_balance[['Date','Ticker','Total Assets','Stockholders Equity','Total Debt','Debt_Ratio']],
    on=['Date','Ticker'], how='inner'
)
if 'Stockholders Equity' in dfm.columns:
    dfm['ROE'] = (dfm['Net Income'] / dfm['Stockholders Equity'] * 100).round(2)
if 'Total Assets' in dfm.columns:
    dfm['ROA'] = (dfm['Net Income'] / dfm['Total Assets'] * 100).round(2)
    dfm['Asset_Turnover'] = (dfm['Total Revenue'] / dfm['Total Assets']).round(3)
aapl = dfm[dfm['Ticker']=='AAPL'].sort_values('Date')
kpis = [c for c in ['Quarter','Gross_Margin','Net_Margin','ROE','ROA','Asset_Turnover'] if c in aapl.columns]
print('Apple KPIs (Last 5Q):')
print(aapl[kpis].tail(5).to_string(index=False))

In [ ]:
df_s = df_income.sort_values(['Ticker','Date'])
df_s['Revenue_QoQ'] = df_s.groupby('Ticker')['Total Revenue'].pct_change(1).mul(100).round(2)
gc = [c for c in ['Quarter','Total Revenue','Revenue_YoY','Revenue_QoQ'] if c in df_s.columns]
print('Apple Growth:')
print(df_s[df_s['Ticker']=='AAPL'][gc].tail(8).to_string(index=False))

In [ ]:
conn = sqlite3.connect(DB_PATH)
q1 = 'SELECT Quarter,Company,ROUND("Total Revenue",2) AS Revenue_Bn,ROUND(Gross_Margin,1) AS Gross_Pct,ROUND(Net_Margin,1) AS Net_Pct FROM income_stmt WHERE Ticker="AAPL" ORDER BY Date DESC LIMIT 8'
print('Apple Last 8Q:')
print(pd.read_sql(q1, conn).to_string(index=False))
q2 = 'SELECT Company,Ticker,Quarter,ROUND(Gross_Margin,1) AS Gross_Pct FROM income_stmt WHERE Date=(SELECT MAX(Date) FROM income_stmt AS s WHERE s.Ticker=income_stmt.Ticker) ORDER BY Gross_Margin DESC'
print('\nGross Margin Ranking:')
print(pd.read_sql(q2, conn).to_string(index=False))
conn.close()

In [ ]:
li=df_income.sort_values('Date').groupby('Ticker').tail(1)
lb=df_balance.sort_values('Date').groupby('Ticker').tail(1)
lc=df_cashflow.sort_values('Date').groupby('Ticker').tail(1)
comps=pd.merge(li[['Ticker','Company','Quarter','Total Revenue','Gross_Margin','Net_Margin','Op_Margin','Revenue_YoY']],lb[['Ticker','Total Assets','Debt_Ratio']],on='Ticker',how='left')
comps=pd.merge(comps,lc[['Ticker','Free Cash Flow']],on='Ticker',how='left')
cd=comps[['Company','Quarter','Total Revenue','Gross_Margin','Op_Margin','Net_Margin','Revenue_YoY','Free Cash Flow','Debt_Ratio']].sort_values('Gross_Margin',ascending=False)
cd.columns=['Company','Quarter','Rev($B)','Gross%','Op%','Net%','YoY%','FCF($B)','Debt%']
print('='*90); print('PEER COMPARISON TABLE'); print('='*90)
print(cd.to_string(index=False))
cd.to_csv('comps_table.csv',index=False)
print('\n✅ comps_table.csv saved')

---
## 📈 SECTION 4 — Visualization
### Part A — [Plotly] Apple Interactive Dashboard

In [ ]:
ai=df_income[df_income['Ticker']=='AAPL'].sort_values('Date')
ab=df_balance[df_balance['Ticker']=='AAPL'].sort_values('Date')
ac=df_cashflow[df_cashflow['Ticker']=='AAPL'].sort_values('Date')
ah=df_prices[df_prices['Ticker']=='AAPL'].sort_values('Date')
h6=ah[ah['Date']>=ah['Date'].max()-pd.Timedelta(days=180)]
Q=ai['Quarter'].tolist()
fig=make_subplots(rows=4,cols=3,
    subplot_titles=['QUARTERLY REVENUE (USD Bn)','GROSS/OP/NET MARGIN (%)','OPERATING INCOME (USD Bn)',
                    'FREE CASH FLOW (USD Bn)','REVENUE YoY GROWTH (%)','EBITDA (USD Bn)',
                    'ASSETS vs EQUITY (USD Bn)','DEBT RATIO (%)','STOCK PRICE 3Y (USD)','OHLC 6M','',''],
    specs=[[{'type':'bar'},{'type':'scatter'},{'type':'bar'}],
           [{'type':'bar'},{'type':'bar'},{'type':'bar'}],
           [{'type':'bar'},{'type':'scatter'},{'type':'scatter'}],
           [{'colspan':3,'type':'candlestick'},None,None]],
    vertical_spacing=0.09,horizontal_spacing=0.06,row_heights=[0.22,0.22,0.22,0.34])
rev=ai['Total Revenue'].values
fig.add_trace(go.Bar(x=Q,y=rev,marker_color=[GOLD if v==max(rev) else BLUE for v in rev],
    marker_line_width=0,showlegend=False,text=[f'{v:.1f}' for v in rev],
    textposition='outside',textfont=dict(size=9,color='white',family='Courier New')),row=1,col=1)
for col,nm,c,d in [('Gross_Margin','Gross',GREEN,'solid'),('Op_Margin','Op',GOLD,'dash'),('Net_Margin','Net',PURPLE,'dot')]:
    if col in ai.columns:
        fig.add_trace(go.Scatter(x=Q,y=ai[col].values,name=nm,mode='lines+markers',
            line=dict(color=c,width=2.5,dash=d),marker=dict(size=6)),row=1,col=2)
if 'Operating Income' in ai.columns:
    op=ai['Operating Income'].values
    fig.add_trace(go.Bar(x=Q,y=op,marker_color=[GREEN if v>=0 else RED for v in op],
        marker_line_width=0,showlegend=False,text=[f'{v:.1f}' for v in op],
        textposition='outside',textfont=dict(size=9,color='white',family='Courier New')),row=1,col=3)
if 'Free Cash Flow' in ac.columns:
    fcf=ac['Free Cash Flow'].values
    fig.add_trace(go.Bar(x=ac['Quarter'].tolist(),y=fcf,
        marker_color=[GREEN if v>=0 else RED for v in fcf],marker_line_width=0,showlegend=False,
        text=[f'{v:.1f}' for v in fcf],textposition='outside',
        textfont=dict(size=9,color='white',family='Courier New')),row=2,col=1)
if 'Revenue_YoY' in ai.columns:
    yoy=ai['Revenue_YoY'].fillna(0).values
    fig.add_trace(go.Bar(x=Q,y=yoy,marker_color=[GREEN if v>=0 else RED for v in yoy],
        marker_line_width=0,showlegend=False,text=[f'{v:.1f}%' for v in yoy],
        textposition='outside',textfont=dict(size=9,color='white',family='Courier New')),row=2,col=2)
if 'EBITDA' in ai.columns:
    eb=ai['EBITDA'].dropna().values
    fig.add_trace(go.Bar(x=Q[:len(eb)],y=eb,
        marker_color=[GOLD if v==max(eb) else '#2A4A7F' for v in eb],marker_line_width=0,showlegend=False,
        text=[f'{v:.1f}' for v in eb],textposition='outside',
        textfont=dict(size=9,color='white',family='Courier New')),row=2,col=3)
if 'Total Assets' in ab.columns:
    fig.add_trace(go.Bar(x=ab['Quarter'].tolist(),y=ab['Total Assets'].values,
        name='Assets',marker_color='rgba(74,158,255,0.6)',marker_line_width=0),row=3,col=1)
if 'Stockholders Equity' in ab.columns:
    fig.add_trace(go.Bar(x=ab['Quarter'].tolist(),y=ab['Stockholders Equity'].values,
        name='Equity',marker_color='rgba(201,168,76,0.8)',marker_line_width=0),row=3,col=1)
if 'Debt_Ratio' in ab.columns:
    fig.add_trace(go.Scatter(x=ab['Quarter'].tolist(),y=ab['Debt_Ratio'].values,
        name='Debt%',mode='lines+markers+text',line=dict(color=RED,width=2.5),
        marker=dict(size=8),fill='tozeroy',fillcolor='rgba(224,82,82,0.08)',
        text=[f'{v:.0f}%' for v in ab['Debt_Ratio'].values],textposition='top center',
        textfont=dict(size=8,family='Courier New')),row=3,col=2)
fig.add_trace(go.Scatter(x=ah['Date'],y=ah['Close'],name='Price',mode='lines',
    line=dict(color=GOLD,width=2),fill='tozeroy',fillcolor='rgba(201,168,76,0.06)'),row=3,col=3)
fig.add_trace(go.Candlestick(x=h6['Date'],open=h6['Open'],high=h6['High'],low=h6['Low'],close=h6['Close'],
    name='OHLC',increasing=dict(line=dict(color=GREEN,width=1),fillcolor='rgba(46,204,113,0.6)'),
    decreasing=dict(line=dict(color=RED,width=1),fillcolor='rgba(224,82,82,0.6)')),row=4,col=1)
cp=ah['Close'].iloc[-1]
fig.update_layout(
    title=dict(text=f"<b>APPLE INC.  ·  AAPL</b><br><span style='font-size:12px;color:#5A6070;font-family:Courier New'>NASDAQ  |  Price: ${cp:,.2f}  |  USD</span>",
               font=dict(size=20,color=GOLD,family='Georgia, serif'),x=0.01,xanchor='left',y=0.995),
    height=1400,template='plotly_dark',paper_bgcolor=BG,plot_bgcolor=SURFACE,
    font=dict(color='#E8EAF0',family='Courier New',size=10),barmode='group',
    legend=dict(orientation='h',y=1.005,x=0.5,xanchor='center',bgcolor='rgba(0,0,0,0)',font=dict(size=9,color='#5A6070')),
    margin=dict(t=90,b=50,l=55,r=40),xaxis_rangeslider_visible=False)
for ann in fig.layout.annotations:
    ann.font.size=9; ann.font.color='#5A6070'; ann.font.family='Courier New'
fig.show()
print('✅ Plotly dashboard done')

### Part B — [Matplotlib] Peer Comparison

In [ ]:
TICKERS=['AAPL','MSFT','GOOGL','META','NVDA']
PAL=['#C9A84C','#4A9EFF','#2ECC71','#A78BFA','#1ABC9C']
fig,axes=plt.subplots(2,2,figsize=(18,12))
fig.patch.set_facecolor('#0A0C10')
fig.suptitle('US BIG TECH — PEER COMPARISON',fontsize=18,color='#C9A84C',fontweight='bold',y=1.01,fontfamily='serif')
def sax(ax,t):
    ax.set_facecolor('#111318'); ax.set_title(t,color='#9AA0B0',fontsize=11,pad=10)
    ax.tick_params(colors='#5A6070',labelsize=8); ax.grid(axis='y',alpha=0.15)
def get(t,df): return df[df['Ticker']==t].sort_values('Date').tail(8)
ax=axes[0,0]; sax(ax,'QUARTERLY REVENUE (USD Bn)')
x=np.arange(8); w=0.15
for i,(t,c) in enumerate(zip(TICKERS,PAL)):
    v=get(t,df_income)['Total Revenue'].values
    p=8-len(v); v=np.pad(v,(p,0),constant_values=np.nan) if p>0 else v[-8:]
    ax.bar(x+i*w,v,width=w,color=c,alpha=0.85,label=t)
ax.legend(fontsize=8,facecolor='#111318',edgecolor='#1E2330',labelcolor='#E8EAF0')
ax=axes[0,1]; sax(ax,'GROSS MARGIN TREND (%)')
for t,c in zip(TICKERS,PAL):
    d=get(t,df_income)
    if 'Gross_Margin' in d.columns and len(d)>0:
        ax.plot(range(len(d)),d['Gross_Margin'].values,color=c,lw=2.5,marker='o',ms=5,label=t)
ax.legend(fontsize=8,facecolor='#111318',edgecolor='#1E2330',labelcolor='#E8EAF0'); ax.grid(alpha=0.15)
ax=axes[1,0]; sax(ax,'NET MARGIN — LATEST QUARTER (%)')
lat=df_income.sort_values('Date').groupby('Ticker').tail(1)
lat=lat[lat['Ticker'].isin(TICKERS)]
if 'Net_Margin' in lat.columns:
    bars=ax.barh(lat['Company'].values,lat['Net_Margin'].values,color=PAL[:len(lat)],alpha=0.85)
    for bar,val in zip(bars,lat['Net_Margin'].values):
        ax.text(bar.get_width()+0.3,bar.get_y()+bar.get_height()/2,f'{val:.1f}%',
                va='center',color='#E8EAF0',fontsize=10,fontfamily='monospace')
ax.grid(axis='x',alpha=0.15)
ax=axes[1,1]; sax(ax,'STOCK PRICE 1Y NORMALIZED (Base=100)')
for t,c in zip(TICKERS,PAL):
    d=df_prices[df_prices['Ticker']==t].sort_values('Date').tail(252)
    if len(d)>0:
        norm=d['Close']/d['Close'].iloc[0]*100
        ax.plot(range(len(d)),norm.values,color=c,lw=2,label=t,alpha=0.9)
ax.axhline(100,color='#5A6070',ls='--',lw=0.8,alpha=0.5)
ax.legend(fontsize=8,facecolor='#111318',edgecolor='#1E2330',labelcolor='#E8EAF0'); ax.grid(alpha=0.15)
plt.tight_layout(pad=2.5)
plt.savefig('peer_comparison.png',dpi=150,bbox_inches='tight',facecolor='#0A0C10',edgecolor='none')
plt.show()
print('✅ peer_comparison.png saved')

### Part C — [Matplotlib] Apple Deep Dive Report

In [ ]:
ai=df_income[df_income['Ticker']=='AAPL'].sort_values('Date').tail(10)
ac=df_cashflow[df_cashflow['Ticker']=='AAPL'].sort_values('Date').tail(10)
ab=df_balance[df_balance['Ticker']=='AAPL'].sort_values('Date').tail(10)
Q=ai['Quarter'].tolist()
fig=plt.figure(figsize=(20,14)); fig.patch.set_facecolor('#0A0C10')
gs=gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)
fig.text(0.02,0.97,'APPLE INC.  ·  AAPL  ·  NASDAQ',fontsize=20,color='#C9A84C',fontweight='bold',fontfamily='serif')
fig.text(0.02,0.94,'Accounting Data Analytics with Python @ UIUC Coursera',fontsize=10,color='#5A6070',fontfamily='monospace')
def sax(ax,t):
    ax.set_facecolor('#111318'); ax.set_title(t,color='#9AA0B0',fontsize=10,pad=8,fontfamily='monospace')
    ax.tick_params(colors='#5A6070',labelsize=8); ax.grid(axis='y',color='#1E2330',alpha=0.8); ax.spines[:].set_edgecolor('#1E2330')
ax1=fig.add_subplot(gs[0,0]); sax(ax1,'QUARTERLY REVENUE (USD Bn)')
bars=ax1.bar(range(len(Q)),ai['Total Revenue'].values,color='#4A9EFF',alpha=0.85,width=0.65)
bars[-1].set_color('#C9A84C')
ax1.set_xticks(range(len(Q))); ax1.set_xticklabels(Q,rotation=45,ha='right',fontsize=7)
ax2=fig.add_subplot(gs[0,1]); sax(ax2,'MARGIN ANALYSIS (%)')
for col,lbl,c,ls in [('Gross_Margin','Gross','#2ECC71','-'),('Op_Margin','Op','#C9A84C','--'),('Net_Margin','Net','#A78BFA',':')]:
    if col in ai.columns:
        ax2.plot(range(len(Q)),ai[col].values,color=c,lw=2.5,ls=ls,marker='o',ms=5,label=lbl)
ax2.set_xticks(range(len(Q))); ax2.set_xticklabels(Q,rotation=45,ha='right',fontsize=7)
ax2.legend(fontsize=8,facecolor='#111318',edgecolor='#1E2330',labelcolor='#E8EAF0'); ax2.grid(alpha=0.2)
ax3=fig.add_subplot(gs[0,2]); sax(ax3,'FREE CASH FLOW (USD Bn)')
if 'Free Cash Flow' in ac.columns:
    fv=ac['Free Cash Flow'].values
    ax3.bar(range(len(fv)),fv,color=['#2ECC71' if v>=0 else '#E05252' for v in fv],alpha=0.85,width=0.65)
    ax3.set_xticks(range(len(ac))); ax3.set_xticklabels(ac['Quarter'].tolist(),rotation=45,ha='right',fontsize=7)
ax3.axhline(0,color='#5A6070',lw=0.8)
ax4=fig.add_subplot(gs[1,0]); sax(ax4,'REVENUE YoY GROWTH (%)')
if 'Revenue_YoY' in ai.columns:
    yoy=ai['Revenue_YoY'].fillna(0).values
    ax4.bar(range(len(Q)),yoy,color=['#2ECC71' if v>=0 else '#E05252' for v in yoy],alpha=0.85,width=0.65)
    ax4.set_xticks(range(len(Q))); ax4.set_xticklabels(Q,rotation=45,ha='right',fontsize=7)
ax4.axhline(0,color='#5A6070',lw=0.8)
ax5=fig.add_subplot(gs[1,1]); sax(ax5,'ASSETS vs EQUITY (USD Bn)')
xb=np.arange(len(ab))
if 'Total Assets' in ab.columns:
    ax5.bar(xb-0.2,ab['Total Assets'].values,0.38,color='#4A9EFF',alpha=0.6,label='Assets')
if 'Stockholders Equity' in ab.columns:
    ax5.bar(xb+0.2,ab['Stockholders Equity'].values,0.38,color='#C9A84C',alpha=0.8,label='Equity')
ax5.set_xticks(xb); ax5.set_xticklabels(ab['Quarter'].tolist(),rotation=45,ha='right',fontsize=7)
ax5.legend(fontsize=8,facecolor='#111318',edgecolor='#1E2330',labelcolor='#E8EAF0')
ax6=fig.add_subplot(gs[1,2]); sax(ax6,'DEBT RATIO (%)')
if 'Debt_Ratio' in ab.columns:
    ax6.fill_between(range(len(ab)),ab['Debt_Ratio'].values,color='#E05252',alpha=0.3)
    ax6.plot(range(len(ab)),ab['Debt_Ratio'].values,color='#E05252',lw=2.5,marker='o',ms=5)
    ax6.set_xticks(range(len(ab))); ax6.set_xticklabels(ab['Quarter'].tolist(),rotation=45,ha='right',fontsize=7)
ax7=fig.add_subplot(gs[2,:]); sax(ax7,'STOCK PRICE — 3 YEAR (USD)')
ap=df_prices[df_prices['Ticker']=='AAPL'].sort_values('Date')
ax7.fill_between(ap['Date'],ap['Close'],color='#C9A84C',alpha=0.15)
ax7.plot(ap['Date'],ap['Close'],color='#C9A84C',lw=2)
ax7.grid(axis='both',alpha=0.15)
plt.savefig('apple_financial_report.png',dpi=150,bbox_inches='tight',facecolor='#0A0C10',edgecolor='none')
plt.show()
print('✅ apple_financial_report.png saved')
print('\n🎉 ALL DONE!')
print('   comps_table.csv')
print('   peer_comparison.png')
print('   apple_financial_report.png')